# شام — مسار جمع وترميز الفيديو (بلا حاجة لـ GPU)

**فرق مهم عن دفتري الصوت والصورة الشقيقين، ذُكر بصراحة كي لا يُفهم خطأً:** الفيديو **لا يملك أداة ترميز (tokenizer) خاصة به إطلاقاً** — هو ببساطة سلسلة من رموز الصورة نفسها، إطاراً إطاراً (`video_tokenizer.py`). لذلك هذا الدفتر **لا يدرّب شيئاً جديداً** — وظيفته فقط: جمع فيديوهات حقيقية، استخراج إطارات حقيقية منها عبر ffmpeg، ثم تحويلها لتسلسل رموز عبر أداة ترميز الصورة **المدرَّبة مسبقاً** (من دفتر `sham_image_tokenizer_track`)، ونشر النتيجة كمجموعة بيانات نصية-رمزية جاهزة لمرحلة تدريب الفيديو القادمة.

**نطاق حقيقي يجب معرفته بصراحة:** لا يوجد مصدر فيديو عربي مكافئ متاح بسهولة بنفس حجم/جاهزية `UCF101` — يستخدم هذا الدفتر مجموعة فيديو حقيقية معروفة للأبحاث (`sayakpaul/ucf101-subset`)، لأن المطلوب هنا هو تنوّع حركة/مشاهد حقيقية لتعلّم البنية الزمنية، لا لغة التعليق (لا توجد تعليقات نصية تُستخدَم في هذه المرحلة أصلاً).

## قبل "Save Version → Save & Run All":
1. **فعّل الإنترنت** من Settings (لا حاجة لـ GPU).
2. تأكد من وجود نفس أسرار Kaggle: `GITHUB_TOKEN`، `KAGGLE_USERNAME`، `KAGGLE_KEY`.
3. **أضف نتاج دفتر `sham_image_tokenizer_track` كمدخل هنا** (+ Add Input → Notebook Output) — **إلزامي وليس اختيارياً**: بلا أداة ترميز صورة مُدرَّبة فعلياً، الرموز الناتجة عن هذا الدفتر عشوائية بلا معنى.
4. **من التشغيل الثاني فصاعداً**: أضف نتاج هذا الدفتر نفسه أيضاً كمدخل ليكمل تجميع الفيديوهات الجديدة فوق ما سبق.
5. استخدم **Save Version → Save & Run All (Commit)** دائماً، ويمكن جدولته للتشغيل التلقائي (Schedule this notebook to run) — بلا معالج رسومي، فلا حصة أسبوعية تحدّه.


### 0) تهيئة البيئة ورفع قيود الذاكرة والموارد (Performance & Resource Setup)


In [ ]:
import resource
import sys
import os

# رفع حد الملفات المفتوحة والموارد لضمان عمل المعالجة دون اختناق البيئة
try:
    resource.setrlimit(resource.RLIMIT_NOFILE, (65536, 65536))
    print("تم رفع قيود عدد الملفات المفتوحة بنجاح إلى 65536.")
except Exception as e:
    print(f"تنبيه عند رفع قيود النظام: {e}")

# ضبط حدود الاستدلال المباشر بالذاكرة
sys.setrecursionlimit(10000)
os.environ["PYTHONUNBUFFERED"] = "1"
print("تم ضبط حدود الاستدلال وإلغاء تخزين المخرجات المؤقت بنجاح.")


### 1) سحب الكود الحقيقي من GitHub


In [ ]:
import os, sys, subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not Path(CLONE_DIR).exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=False)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert Path(CODE_DIR, "model.py").exists()
sys.path.insert(0, CODE_DIR)
print("كود شام:", CODE_DIR)

### 2) تثبيت المكتبات الإضافية


In [ ]:
import subprocess, sys
for p in ["telethon", "nest_asyncio", "opencv-python-headless", "Pillow",
          "datasets", "pydub", "soundfile", "imageio-ffmpeg", "yt-dlp"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)
print("تم التثبيت")

3 اعلام المسار

In [ ]:
import os
os.environ["SHAM_TRACK"] = "video_frames_and_audio"
os.environ["SHAM_ALLOW_ADULT_VIDEO"] = "1"
TARGET_VIDEOS = 60
print("الهدف:", TARGET_VIDEOS, "فيديو")

4— تحميل Image + Audio Tokenizer (النسخة القوية)

In [ ]:
import sys
import json as _json
from pathlib import Path
import torch

# 1. إدراج كافة المسارات المحتملة لملفات الـ Tokenizers في sys.path
possible_paths = [
    "/kaggle/working",
    "/kaggle/working/Ttbik",
    "/kaggle/working/Ttbik/ai-system/scripts",
    "/kaggle/working/Ttbik/ai-system/colab/sham_small"
]
for p in possible_paths:
    if p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# البحث الديناميكي عن مكان ملف image_tokenizer.py وإضافة مجلده تلقائياً
for py_file in Path("/kaggle/working").rglob("image_tokenizer.py"):
    parent_dir = str(py_file.parent)
    if parent_dir not in sys.path:
        sys.path.insert(0, parent_dir)

# 2. استدعاء الوحدات الآن بعد تأمين المسارات
from image_tokenizer import ImageTokenizer, ImageTokenizerConfig
from audio_tokenizer import AudioTokenizer, AudioTokenizerConfig

def load_tokenizer_robust(path, TokenizerClass, ConfigClass, expected_num_codes, strict=False):
    payload = torch.load(path, map_location="cpu", weights_only=False)
    step = 0
    samples = 0
    
    if not isinstance(payload, dict):
        state_dict = payload
        cfg_data = {}
    elif "state_dict" in payload:
        state_dict = payload["state_dict"]
        step = int(payload.get("step", 0))
        samples = int(payload.get("samples", 0))
        cfg_data = payload.get("config", {})
    else:
        state_dict = payload
        cfg_data = {}

    try:
        if isinstance(cfg_data, dict) and cfg_data:
            cfg = ConfigClass(**cfg_data)
        else:
            cfg = ConfigClass(num_codes=expected_num_codes)
    except Exception:
        cfg = ConfigClass(num_codes=expected_num_codes)

    tokenizer = TokenizerClass(cfg)
    
    try:
        tokenizer.load_state_dict(state_dict, strict=strict)
    except Exception as e:
        print(f"⚠️ تحذير أثناء تحميل الأوزان من {path.name}: {e}")
        try:
            tokenizer.load_state_dict(state_dict, strict=False)
            print("✅ تم التحميل بنجاح في وضع strict=False")
        except Exception:
            print("⚠️ غير متوافق تماماً → إنشاء كائن جديد")
            tokenizer = TokenizerClass(ConfigClass(num_codes=expected_num_codes))
            step = 0
            samples = 0

    return tokenizer, step, samples


# ----- صورة (Image Tokenizer) -----
img_ckpts = sorted(Path("/kaggle/input").rglob("image_tokenizer.pt"), key=lambda p: p.stat().st_mtime)

if not img_ckpts:
    print("⚠️ لا يوجد image_tokenizer.pt → إنشاء جديد")
    image_tokenizer = ImageTokenizer(ImageTokenizerConfig(num_codes=8192))
    image_start_step = 0
    image_samples = 0
else:
    try:
        image_tokenizer, image_start_step, image_samples = load_tokenizer_robust(
            img_ckpts[-1], ImageTokenizer, ImageTokenizerConfig, 8192, strict=False
        )
        print(f"✅ ImageTokenizer محمّل بنجاح من: {img_ckpts[-1].name}")
    except Exception as e:
        print(f"⚠️ فشل التحميل ({e}) → إنشاء جديد")
        image_tokenizer = ImageTokenizer(ImageTokenizerConfig(num_codes=8192))
        image_start_step = 0
        image_samples = 0

print(f"Image | step={image_start_step} | samples={image_samples}")


# ----- صوت (Audio Tokenizer) -----
audio_ckpts = sorted(Path("/kaggle/input").rglob("audio_tokenizer.pt"), key=lambda p: p.stat().st_mtime)

if audio_ckpts:
    try:
        audio_tokenizer, audio_start_step, audio_samples = load_tokenizer_robust(
            audio_ckpts[-1], AudioTokenizer, AudioTokenizerConfig, 2048, strict=False
        )
        print(f"✅ AudioTokenizer محمّل بنجاح من: {audio_ckpts[-1].name}")
    except Exception as e:
        print(f"⚠️ فشل التحميل ({e}) → إنشاء جديد")
        audio_tokenizer = AudioTokenizer(AudioTokenizerConfig(num_codes=2048))
        audio_start_step = 0
        audio_samples = 0
else:
    print("⚠️ لا يوجد audio_tokenizer.pt → إنشاء جديد")
    audio_tokenizer = AudioTokenizer(AudioTokenizerConfig(num_codes=2048))
    audio_start_step = 0
    audio_samples = 0

print(f"Audio | step={audio_start_step} | samples={audio_samples}")


5-حفظ كوكيز x

In [ ]:
cookies_content = r"""# Netscape HTTP Cookie File
# http://curl.haxx.se/rfc/cookie_spec.html
# This file was generated by Cookie-Editor
#HttpOnly_.x.com	TRUE	/	TRUE	1821657157	auth_token	5441f2270c4a7a21bd9f96eaef371257012f50ce
.x.com	TRUE	/	TRUE	1790129905	gt	2102545581625201075
x.com	FALSE	/	FALSE	1824681160	__cuid	5535cb3b-f0ba-41c3-acaf-08742221db69
.x.com	TRUE	/	TRUE	1824680905	guest_id	v1%3A179012090470385459
.x.com	TRUE	/	TRUE	1821657163	twid	u%3D1778818165339836416
x.com	FALSE	/	FALSE	1805672920	g_state	{"i_l":1,"i_ll":1790120920450,"i_b":"QQ8Jjoy+LiZDzAD7CQF6jtmHStmEnODipedQxwcD/B0","i_e":{"enable_itp_optimization":24},"i_et":1790120920450}
x.com	FALSE	/	FALSE	1790207817	lang	ar
#HttpOnly_.x.com	TRUE	/	TRUE	1790122958	__cf_bm	8jv7L0J2bCrgY86xHAf2fBkA9H_HTqZJ13s.sugxyW4-1790121158.1384737-1.0.1.1-6jcqEbXTDTmAAfG83gf7Yct.sHFcjMI4cR2jq7aV8hEs1RchkLVtX6np9FzrJeQacl6JYj7YpA_4hNX5Lb6hEEakyJ9Ujk9VeB.y5gdU7TBIlTwP6GZ.5qag0g8o1cTP
.x.com	TRUE	/	FALSE	1824681160	__cuid	5535cb3b-f0ba-41c3-acaf-08742221db69
.x.com	TRUE	/	TRUE	1824681157	ct0	5465dc74191d8c221bd78cb2582fd5a5792eb7d68606771c948ada2d4cbca1bc3ebe8469a9c7ad7286a7eeeca3a330dbd78fa4175b47a27238128e21d39b1663c4a4bd4993f2669c6597d780d8707fe6
.x.com	TRUE	/	TRUE	1824681163	guest_id_ads	v1%3A179012090470385459
.x.com	TRUE	/	TRUE	1824681163	guest_id_marketing	v1%3A179012090470385459
.x.com	TRUE	/	TRUE	1824680910	personalization_id	"v1_3EqFikqkjNUtZxaS+cseDw=="
"""

with open("/kaggle/working/x_cookies.txt", "w", encoding="utf-8") as f:
    f.write(cookies_content.strip())

print("✅ تم حفظ كوكيز X")

6-سحب الفيديوهات من X بالهاشتاجات

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from urllib.parse import quote

# 1. تثبيت أداة gallery-dl المتخصصة في البحث المباشر بـ X/Twitter
print("📦 جاري تثبيت أداة gallery-dl للبحث والتنزيل المباشر...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "gallery-dl"], check=False)

VIDEO_DIR = Path("/kaggle/working/videos")
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

if 'TARGET_VIDEOS' not in globals():
    TARGET_VIDEOS = 50

HASHTAGS = [
    # إنجليزي
    "hotwife",
    "swingers",
    "cuckold",
    "swinger",
    "hotwifesharing",
    
    # تركي
    "eşdeğiştirme",
    "swingertürkiye",
    "cuckoldtürkiye",
    "swingerçift",
    
    # عربي
    "تبادل_زوجات",
    "سكس_عربي",
]

MAX_PER_HASHTAG = 8
cookies_file = "/kaggle/working/x_cookies.txt"

print("🚀 بدء البحث والسحب المباشر الحقيقي من X (Twitter)...")

for tag in HASHTAGS:
    # الحصول على إجمالي الفيديوهات المسحوبة فعلياً
    current_files = list(VIDEO_DIR.glob("**/*.mp4")) + list(VIDEO_DIR.glob("**/*.webm")) + list(VIDEO_DIR.glob("**/*.mkv"))
    if len(current_files) >= TARGET_VIDEOS:
        print(f"🎯 تم الوصول للعدد المستهدف من الفيديوهات الحقيقية: {len(current_files)}")
        break
    
    print(f"\n🔍 بحث حقيقي في X عن الهاشتاج: #{tag}")
    
    # رابط استعلام البحث المباشر في X للفيديوهات فقط
    search_url = f"https://x.com/search?q=%23{quote(tag)}%20filter%3Avideos&f=live"
    
    # بناء أمر gallery-dl للبحث وتنزيل مقاطع الفيديو فقط
    cmd = [
        "gallery-dl",
        "-d", str(VIDEO_DIR),
        "--range", f"1-{MAX_PER_HASHTAG}",
        "--filter", "extension in ('mp4', 'webm', 'mkv', 'mov')",
        "-o", "extractor.twitter.videos=true",
        "-o", "extractor.twitter.images=false",
        search_url
    ]
    
    if Path(cookies_file).exists():
        cmd.extend(["--cookies", cookies_file])
    else:
        print("⚠️ تنبيه: ملف الكوكيز x_cookies.txt غير موجود. البحث في X يتطلب تسجيل دخول (كوكيز).")

    try:
        # تنفيذ التنزيل الفعلي
        subprocess.run(cmd, check=False, timeout=120)
    except Exception as e:
        print(f"❌ خطأ أثناء تنفيذ البحث: {str(e)[:80]}")

# 2. حصر وتنظيم كافة الفيديوهات المسحوبة حقيقياً
video_files = sorted(
    list(VIDEO_DIR.glob("**/*.mp4")) + 
    list(VIDEO_DIR.glob("**/*.webm")) + 
    list(VIDEO_DIR.glob("**/*.mkv"))
)

print("\n" + "="*50)
print(f"📊 نتائج البحث الفعلي والملفات المباشرة:")
print(f"✅ إجمالي الفيديوهات الحقيقية المسحوبة من X: {len(video_files)}")

if len(video_files) > 0:
    print("📁 قائمة الفيديوهات الحقيقية المحفوظة:")
    for v in video_files[:10]:
        print(f"  • {v.name}")
    if len(video_files) > 10:
        print(f"  ... و {len(video_files) - 10} فيديوهات أخرى.")
else:
    print("⚠️ لم يتم العثور على أية فيديوهات حقيقية.")
    print("👉 أسباب محتملة:")
    print("   1. ملف x_cookies.txt منتهي الصلاحية أو تحتاج لإعادة تصديره من متصفحك.")
    print("   2. قيود منصة X على الحساب المستخدم في الكوكيز.")


استخراج الإطارات (Image)

In [ ]:
import cv2
from PIL import Image
from pathlib import Path
import shutil

FRAMES_DIR = Path("/kaggle/working/frames")
if FRAMES_DIR.exists():
    shutil.rmtree(FRAMES_DIR)
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

MAX_FRAMES_PER_VIDEO = 8
IMAGE_SIZE = getattr(image_tokenizer.cfg, "image_size", 256)
frame_paths = []

print(f"استخراج إطارات من {len(video_files)} فيديو...")

for vi, vpath in enumerate(video_files):
    try:
        cap = cv2.VideoCapture(str(vpath))
        if not cap.isOpened():
            continue
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        step = max(1, total // MAX_FRAMES_PER_VIDEO) if total > MAX_FRAMES_PER_VIDEO else 8
        got, idx = 0, 0
        while got < MAX_FRAMES_PER_VIDEO:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ok, frame = cap.read()
            if not ok:
                break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            Image.fromarray(rgb).resize((IMAGE_SIZE, IMAGE_SIZE)).save(
                FRAMES_DIR / f"f_{vi:03d}_{got:02d}.png"
            )
            frame_paths.append(FRAMES_DIR / f"f_{vi:03d}_{got:02d}.png")
            got += 1
            idx += step
        cap.release()
        if got > 0:
            print(f"  ✅ {vi+1}/{len(video_files)} → {got} إطار")
    except Exception as e:
        print(f"  ❌ {vpath.name}: {str(e)[:50]}")

print(f"\nإجمالي الإطارات: {len(frame_paths)}")

تدريب Image Tokenizer-8

In [ ]:
import sys
from pathlib import Path

# حقن المسارات المحلية لتفادي أخطاء الاستدعاء
possible_paths = [
    "/kaggle/working",
    "/kaggle/working/Ttbik",
    "/kaggle/working/Ttbik/ai-system/scripts",
    "/kaggle/working/Ttbik/ai-system/colab/sham_small"
]
for p in possible_paths:
    if p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

for py_file in Path("/kaggle/working").rglob("train_image_tokenizer.py"):
    parent_dir = str(py_file.parent)
    if parent_dir not in sys.path:
        sys.path.insert(0, parent_dir)

import torch, numpy as np, time
from PIL import Image
from train_image_tokenizer import train_vqvae as train_image_vqvae

def frame_to_tensor(path, size):
    arr = np.array(Image.open(path).convert("RGB").resize((size, size)), dtype=np.float32) / 255.0
    arr = arr * 2.0 - 1.0
    return torch.from_numpy(arr).permute(2, 0, 1)

if len(frame_paths) == 0:
    raise RuntimeError("لا توجد إطارات للتدريب")

# تحديد عدد محدد من الإطارات لتدريب سريع ومباشر لتفادي تعليق الـ CPU
sample_paths = frame_paths[:min(32, len(frame_paths))]
img_batch = torch.stack([frame_to_tensor(p, IMAGE_SIZE) for p in sample_paths])
print("image batch:", img_batch.shape)

B = 8
epochs = 1

# تدريب مباشر بدون حلقة تجريبية أو حسابات وقت وهمية
img_stats = train_image_vqvae(
    image_tokenizer, 
    img_batch, 
    num_epochs=epochs, 
    batch_size=B, 
    log_every=5
)

print("image loss:", img_stats.epoch_losses[-1] if img_stats and hasattr(img_stats, 'epoch_losses') and img_stats.epoch_losses else "N/A")


9-استخراج صوت + تدريب Audio

In [ ]:
import sys
import subprocess
import torch
import torchaudio
import numpy as np, time, soundfile as sf
from pathlib import Path

# 1. إدراج المسارات ديناميكياً لتفادي ModuleNotFoundError
possible_paths = [
    "/kaggle/working",
    "/kaggle/working/Ttbik",
    "/kaggle/working/Ttbik/ai-system/scripts",
    "/kaggle/working/Ttbik/ai-system/colab/sham_small"
]
for p in possible_paths:
    if p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

for py_file in Path("/kaggle/working").rglob("train_audio_tokenizer.py"):
    parent_dir = str(py_file.parent)
    if parent_dir not in sys.path:
        sys.path.insert(0, parent_dir)

from train_audio_tokenizer import train_vqvae as train_audio_vqvae

AUDIO_DIR = Path("/kaggle/working/audio_from_video")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
wav_files = []

print(f"🚀 استخراج الصوت عبر ffmpeg لـ {len(video_files)} فيديو...")

# 2. استخراج سريع جداً لتفادي بطء pydub
for i, vp in enumerate(video_files):
    out = AUDIO_DIR / f"a_{i:04d}.wav"
    cmd = [
        "ffmpeg", "-y", "-i", str(vp),
        "-vn", "-acodec", "pcm_s16le",
        "-ar", "16000", "-ac", "1",
        str(out)
    ]
    try:
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
        if out.exists() and out.stat().st_size > 1000:
            wav_files.append(out)
    except Exception as e:
        print("audio skip", vp.name, str(e)[:40])

print("ملفات صوت:", len(wav_files))

audio_stats = None
if wav_files:
    def load_wav_segment(path, target_len=16000*3):
        data, sr = sf.read(str(path), dtype="float32")
        if data.ndim > 1:
            data = data.mean(axis=1)
        if len(data) > target_len:
            start = np.random.randint(0, len(data) - target_len)
            data = data[start:start+target_len]
        else:
            data = np.pad(data, (0, max(0, target_len - len(data))))
        return torch.from_numpy(data).float()

    audio_tensors = [load_wav_segment(p) for p in wav_files[:min(30, len(wav_files))]]
    if audio_tensors:
        raw_audio = torch.stack(audio_tensors) # [30, 48000]
        
        # 3. تحويل الموجة الصوتية إلى Mel-Spectrogram
        mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=16000,
            n_fft=1024,
            hop_length=256,
            n_mels=80
        )
        
        audio_batch = mel_transform(raw_audio).unsqueeze(1) # [30, 1, 80, 188]
        
        # 4. محاذاة البُعد الزمني ليكون مضاعفاً للعدد 16 (188 -> 176) لتطابق أبعاد Reconstruction
        time_len = audio_batch.shape[-1]
        valid_len = (time_len // 16) * 16
        audio_batch = audio_batch[..., :valid_len] # [30, 1, 80, 176]
        
        print("audio batch final shape (Aligned Mel):", audio_batch.shape)

        # 5. تشغيل التدريب
        B = 4
        epochs = 2
        
        audio_stats = train_audio_vqvae(
            audio_tokenizer, 
            audio_batch, 
            num_epochs=epochs, 
            batch_size=B, 
            log_every=1
        )
        print("audio loss:", audio_stats.epoch_losses[-1] if audio_stats and hasattr(audio_stats, 'epoch_losses') and audio_stats.epoch_losses else "N/A")


10-— حفظ النقاط + رفع على Kaggle

In [ ]:
from train_image_tokenizer import save_tokenizer_checkpoint as save_image_ckpt
from train_audio_tokenizer import save_tokenizer_checkpoint as save_audio_ckpt
import shutil, subprocess, os, json as _json
from kaggle_secrets import UserSecretsClient

ckpt = Path("/kaggle/working/checkpoints")
ckpt.mkdir(parents=True, exist_ok=True)

# Image
img_final_step = image_start_step + len(getattr(img_stats, "epoch_losses", []) or [0])
img_final_samples = image_samples + len(frame_paths)
save_image_ckpt(str(ckpt / "image_tokenizer.pt"), image_tokenizer, step=img_final_step)
(ckpt / "image_tokenizer_progress.json").write_text(_json.dumps({
    "samples_consumed": img_final_samples,
    "step": img_final_step,
    "type": "video_frames",
    "num_videos": len(video_files),
    "num_frames": len(frame_paths),
}, ensure_ascii=False, indent=2))

# Audio
if audio_stats is not None:
    aud_final_step = audio_start_step + len(getattr(audio_stats, "epoch_losses", []) or [0])
    aud_final_samples = audio_samples + len(wav_files)
    save_audio_ckpt(str(ckpt / "audio_tokenizer.pt"), audio_tokenizer, step=aud_final_step)
    (ckpt / "audio_tokenizer_progress.json").write_text(_json.dumps({
        "samples_consumed": aud_final_samples,
        "step": aud_final_step,
        "type": "audio_from_video",
    }, ensure_ascii=False, indent=2))

print("ملفات الحفظ:", list(ckpt.iterdir()))

# رفع
secrets = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")
USER = os.environ["KAGGLE_USERNAME"]
SLUG = f"{USER}/sham-video-frames-audio-tokenizers"

upload = Path("/kaggle/working/for_upload")
if upload.exists():
    shutil.rmtree(upload)
upload.mkdir()
for f in ckpt.iterdir():
    shutil.copy2(f, upload / f.name)
(upload / "dataset-metadata.json").write_text(_json.dumps({
    "title": "sham-video-frames-audio-tokenizers",
    "id": SLUG,
    "licenses": [{"name": "unknown"}],
}))

listed = subprocess.run(["kaggle", "datasets", "list", "-m", "--csv"], capture_output=True, text=True)
exists = SLUG in (listed.stdout or "")
cmd = (["kaggle", "datasets", "version", "-p", str(upload), "-m", f"video step {img_final_step}", "-r", "zip"]
       if exists else ["kaggle", "datasets", "create", "-p", str(upload), "-r", "zip"])
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout or r.stderr)
print("exists:", exists)